**In this exercise, we are working with health expenditure data, which is an time series data.**

In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# ===============================
# 1. Load and prepare the dataset
# ===============================
df = pd.read_csv('Data.csv')

year_columns = [col for col in df.columns if col.isdigit()]
data = df[['Country Name'] + year_columns].set_index('Country Name')

# ======================================
# 2. Normalize each country's data (0–1)
# ======================================
scalers = {}
normalized_data = pd.DataFrame(index=data.index, columns=year_columns)

for country in data.index:
    values = data.loc[country].values.reshape(-1, 1)

    if np.isnan(values).any():  # Skip countries with missing data
        continue

    scaler = MinMaxScaler()
    normalized = scaler.fit_transform(values)
    normalized_data.loc[country] = normalized.flatten()
    scalers[country] = scaler

normalized_data = normalized_data.dropna()  # Drop countries still containing NaNs

# ========================================
# 3. Create input/output sequences for RNN
# ========================================
sequence_length = 5
X, y = [], []

for country in normalized_data.index:
    values = normalized_data.loc[country].values.astype(float)

    for i in range(len(values) - sequence_length):
        seq = values[i:i+sequence_length]
        label = values[i+sequence_length]

        if np.isnan(seq).any() or np.isnan(label):
            continue

        X.append(seq)
        y.append(label)

X = np.array(X).reshape(-1, sequence_length, 1)
y = np.array(y)

# ======================================
# 4. Clean NaNs from training data
# ======================================
X_reshaped = X.reshape(X.shape[0], X.shape[1])
nan_rows = np.any(np.isnan(X_reshaped), axis=1)
X_clean = X[~nan_rows]
y_clean = y[~nan_rows]

# ======================================
# 5. Define and train the LSTM model
# ======================================
model = Sequential([
    LSTM(64, activation='relu', input_shape=(sequence_length, 1)),
    Dense(1)
])
model.compile(optimizer=Adam(learning_rate=0.1), loss='mse')
model.fit(X_clean, y_clean, epochs=100, verbose=1)

# ======================================
# 6. Forecast 2022–2026 per country
# ======================================
predictions = {}

for country in normalized_data.index:
    history = normalized_data.loc[country].values.astype(float).tolist()
    future = []

    for _ in range(5):
        input_seq = np.array(history[-sequence_length:]).reshape(1, sequence_length, 1)
        pred = model.predict(input_seq, verbose=0)
        history.append(pred[0, 0])
        future.append(pred[0, 0])

    # Inverse transform to get actual GDP percentages
    inv_future = scalers[country].inverse_transform(np.array(future).reshape(-1, 1)).flatten()
    predictions[country] = inv_future

# ======================================
# 7. Save results
# ======================================
years = [2022, 2023, 2024, 2025, 2026]
forecast_df = pd.DataFrame(predictions, index=years).T
forecast_df.index.name = 'Country'
forecast_df.columns = [str(y) for y in years]


Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1458
Epoch 2/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0428
Epoch 3/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0367
Epoch 4/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0348
Epoch 5/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0316
Epoch 6/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0317
Epoch 7/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0299
Epoch 8/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0303
Epoch 9/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0280
Epoch 10/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0295
Epoch 11/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0281
Epoch 12/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0288
Epoch 13/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0274
Epoch 14/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0287
Epoch 15/100
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0262
Epoch 16/100
88/

In [45]:
#Forecast 2022–2026 per country
forecast_df

,2022,2023,2024,2025,2026
Country,,,,,
United States,16.942709,16.483154,16.368757,16.139603,16.406132
North America,16.542389,16.087330,15.976401,15.751328,16.012829
Liberia,15.606533,14.034847,13.263356,12.527004,12.049931
Palau,16.227596,15.715111,15.099505,14.707760,14.302161
Kiribati,13.919848,13.162357,13.010973,12.684204,13.053878
...,...,...,...,...,...
Djibouti,2.933368,3.039844,3.141592,3.244808,3.362395
Sudan,3.193001,3.496720,3.664368,3.843506,4.020758
Gabon,2.738243,2.781436,2.844761,2.864826,2.930353
